# 06 — Guide Assignment

**Prerequisites:** Run [05_quality_control.ipynb](05_quality_control.ipynb) first. Requires `data/norman_qc.h5ad`.

**What you'll learn:**
- The difference between a raw guide UMI count matrix and a final guide assignment
- Two approaches: threshold-based assignment vs. probabilistic assignment via `pertpy`
- How to handle multiplets (cells with two detected guides)
- Assignment confidence scoring
- Guide concordance: verifying that two guides targeting the same gene agree on the transcriptional effect

**Estimated time:** 2 hours

In [ ]:
import os
import scanpy as sc
import pertpy as pt
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import scipy.sparse

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")

DATA_DIR = "data"
FIG_DIR = "figures"

adata = sc.read_h5ad(os.path.join(DATA_DIR, "norman_qc.h5ad"))
print(adata)

PERT_COL = "perturbation" if "perturbation" in adata.obs.columns else "condition"
print(f"Perturbation column: '{PERT_COL}'")

## 1. Concepts: from guide UMI matrix to guide assignment

In a freshly processed Perturb-seq dataset (directly from Cell Ranger), the guide capture output is a **cells × guides UMI count matrix** — each entry is the number of UMIs from guide `g` in cell `c`.

Assignment converts this matrix into a single label per cell:

```
Raw guide UMI matrix (cells × guides)
┌─────────┬──────┬──────┬──────┬──────┐
│ cell_id │ g1   │ g2   │ g3   │ g4   │
├─────────┼──────┼──────┼──────┼──────┤
│ AAACCC  │  42  │   1  │   0  │   0  │  → assigned to g1
│ AAGCTT  │   0  │  38  │   1  │   0  │  → assigned to g2
│ ATCGGA  │  18  │  22  │   0  │   0  │  → multiplet (g1+g2)
│ CCGATT  │   1  │   0  │   1  │   0  │  → unassigned (too few UMIs)
└─────────┴──────┴──────┴──────┴──────┘
```

Two main assignment strategies:
1. **Threshold-based**: assign to the guide with the most UMIs if ≥ threshold; declare multiplet if second guide also ≥ threshold
2. **Probabilistic** (pertpy): fit a Poisson mixture model to distinguish true captures from ambient/background

## 2. Working with the Norman 2019 labeled dataset

The Norman 2019 h5ad we downloaded already contains guide assignments in `adata.obs`. In this notebook, we:
1. Parse and clean the existing assignment metadata
2. Demonstrate how `pertpy` would re-assign from a guide UMI matrix
3. Add assignment confidence and concordance metrics

In [ ]:
# Parse perturbation labels into structured assignment metadata
def parse_perturbation(pert_label):
    """Parse a Norman-style perturbation label into components."""
    p = str(pert_label)
    if any(kw in p.lower() for kw in ["ctrl", "control", "non", "ntc"]):
        return {
            "gene_target": "NTC",
            "guide_id": p,
            "is_ntc": True,
            "is_dual": False,
            "n_targets": 0,
        }
    elif "+" in p:
        genes = p.split("+")
        return {
            "gene_target": p,
            "guide_id": p,
            "is_ntc": False,
            "is_dual": True,
            "n_targets": len(genes),
        }
    else:
        return {
            "gene_target": p,
            "guide_id": p,
            "is_ntc": False,
            "is_dual": False,
            "n_targets": 1,
        }

parsed = adata.obs[PERT_COL].apply(parse_perturbation).apply(pd.Series)
for col in parsed.columns:
    adata.obs[col] = parsed[col].values

print("Assignment metadata added to obs:")
print(adata.obs[[PERT_COL, "gene_target", "is_ntc", "is_dual", "n_targets"]].head(10))
print(f"\nNTC cells: {adata.obs['is_ntc'].sum()}")
print(f"Single-target cells: {(~adata.obs['is_dual'] & ~adata.obs['is_ntc']).sum()}")
print(f"Dual-target cells: {adata.obs['is_dual'].sum()}")

## 3. Pertpy guide assignment (demonstration)

`pertpy` implements probabilistic guide assignment via `pt.tl.GuideRNAAssignment`. It fits a Poisson mixture model to distinguish guide-positive cells from background.

We demonstrate the API using a synthetic guide UMI matrix. When you have a real Cell Ranger guide capture matrix, this is the tool to use.

In [ ]:
# Construct a synthetic guide UMI AnnData to demonstrate pertpy assignment
np.random.seed(42)
n_cells = 2000
guides = ["CBFB_g1", "CBFB_g2", "RUNX1_g1", "RUNX1_g2", "NTC_1", "NTC_2"]
n_guides = len(guides)

# Simulate: each cell has one 'true' guide with high UMIs, rest are background
true_guide_idx = np.random.choice(n_guides, size=n_cells)
guide_matrix = np.random.poisson(1.0, size=(n_cells, n_guides))  # background
for i, g in enumerate(true_guide_idx):
    guide_matrix[i, g] += np.random.poisson(30)  # true capture signal

# Add some multiplets
multiplet_mask = np.random.rand(n_cells) < 0.05
second_guide = np.random.choice(n_guides, size=n_cells)
guide_matrix[multiplet_mask, second_guide[multiplet_mask]] += np.random.poisson(25, size=multiplet_mask.sum())

guide_adata = anndata.AnnData(
    X=scipy.sparse.csr_matrix(guide_matrix),
    obs=pd.DataFrame(index=[f"cell_{i}" for i in range(n_cells)]),
    var=pd.DataFrame(index=guides)
)
print(f"Synthetic guide UMI AnnData: {guide_adata.shape}")

In [ ]:
# pertpy guide assignment
try:
    gra = pt.tl.GuideRNAAssignment()
    # Check pertpy API — method name may vary by version
    if hasattr(gra, 'assign'):
        guide_adata = gra.assign(guide_adata)
    elif hasattr(gra, 'fit'):
        guide_adata = gra.fit(guide_adata)
    
    print("pertpy assignment complete.")
    print(guide_adata.obs.head())
except Exception as e:
    print(f"Note: pertpy GuideRNAAssignment raised: {e}")
    print("Falling back to manual threshold assignment (see section 4).")

## 4. Threshold-based assignment (manual)

A simple, interpretable approach: assign the cell to the guide with the highest UMI count, provided it exceeds a minimum threshold. Flag cells where a second guide also exceeds the threshold as multiplets.

In [ ]:
def threshold_assign(guide_matrix, guide_names, min_umi=5, multiplet_umi=5):
    """
    Threshold-based guide assignment.
    
    Returns a DataFrame with columns:
      assigned_guide, assigned_gene, top_umi, purity_score, status
    status: 'assigned', 'multiplet', 'unassigned'
    """
    if scipy.sparse.issparse(guide_matrix):
        M = guide_matrix.toarray()
    else:
        M = np.array(guide_matrix)
    
    n_cells = M.shape[0]
    results = []
    
    for i in range(n_cells):
        row = M[i]
        total = row.sum()
        sorted_idx = np.argsort(row)[::-1]
        top_umi = row[sorted_idx[0]]
        second_umi = row[sorted_idx[1]] if len(sorted_idx) > 1 else 0
        purity = top_umi / total if total > 0 else 0
        
        if top_umi < min_umi:
            status = "unassigned"
            assigned = "unassigned"
        elif second_umi >= multiplet_umi:
            status = "multiplet"
            assigned = guide_names[sorted_idx[0]] + "+" + guide_names[sorted_idx[1]]
        else:
            status = "assigned"
            assigned = guide_names[sorted_idx[0]]
        
        results.append({
            "assigned_guide": assigned,
            "top_umi": top_umi,
            "purity_score": purity,
            "status": status,
        })
    
    return pd.DataFrame(results)


assignment_df = threshold_assign(
    guide_adata.X,
    guide_adata.var_names.tolist(),
    min_umi=5,
    multiplet_umi=5,
)
print("Assignment status:")
print(assignment_df["status"].value_counts())
print(f"\nPurity score (assigned cells only):")
print(assignment_df[assignment_df["status"]=="assigned"]["purity_score"].describe())

In [ ]:
# Visualize purity score distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(assignment_df["purity_score"], bins=40, color="steelblue", edgecolor="none")
axes[0].axvline(0.75, color="red", linestyle="--", label="Purity threshold (0.75)")
axes[0].set_xlabel("Guide purity score")
axes[0].set_ylabel("Cells")
axes[0].set_title("Guide purity score distribution")
axes[0].legend()

status_counts = assignment_df["status"].value_counts()
axes[1].pie(status_counts, labels=status_counts.index, autopct="%1.1f%%",
            colors=["#4CAF50", "#FF9800", "#F44336"])
axes[1].set_title("Assignment status")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "06_assignment_quality.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Guide concordance

For genes targeted by multiple guides, **concordance** measures whether different guides produce the same transcriptional effect. High concordance (r > 0.5) strongly suggests on-target activity.

Method:
1. For each guide targeting gene G, compute the mean expression vector across all cells with that guide
2. Subtract the NTC mean (perturbation signature)
3. Compute pairwise Pearson correlation between signatures of different guides for the same gene

In [ ]:
# Use the Norman dataset for concordance analysis
# Work with single-target cells only
single_cells = adata[~adata.obs["is_dual"] & ~adata.obs["is_ntc"]].copy()
ntc_cells = adata[adata.obs["is_ntc"]].copy()

print(f"Single-target cells: {len(single_cells)}")
print(f"NTC cells: {len(ntc_cells)}")

# For concordance we need normalized expression
# Use the expression matrix as-is (may already be normalized)
# If not normalized, normalize here
def get_dense(adata_obj):
    X = adata_obj.X
    if scipy.sparse.issparse(X):
        return X.toarray()
    return np.array(X)

# Compute NTC mean
ntc_mean = get_dense(ntc_cells).mean(axis=0)  # shape: (n_genes,)

In [ ]:
# Compute per-guide mean expression and perturbation signature
guide_signatures = {}

for gene, grp in single_cells.obs.groupby("gene_target"):
    cell_idx = grp.index
    cells_subset = single_cells[cell_idx]
    mean_expr = get_dense(cells_subset).mean(axis=0)
    signature = mean_expr - ntc_mean  # perturbation signature
    guide_signatures[gene] = signature

n_genes_targeted = len(guide_signatures)
print(f"Computed signatures for {n_genes_targeted} single-gene perturbations.")

# In a real dataset, you would compute one signature per guide (not per gene)
# and then correlate guides targeting the same gene.
# Here, each entry in guide_signatures IS one guide (Norman encodes one guide per label).
# For demonstration, we show signature correlations across all single-gene perturbations.

# Select top N perturbations by signature magnitude for visualization
sig_magnitudes = {g: np.abs(sig).mean() for g, sig in guide_signatures.items()}
top_genes = sorted(sig_magnitudes, key=sig_magnitudes.get, reverse=True)[:20]
print(f"\nTop 20 perturbations by signature magnitude: {top_genes}")

In [ ]:
# Compute pairwise correlation of perturbation signatures (top genes)
sig_matrix = np.array([guide_signatures[g] for g in top_genes])

# Pearson correlation matrix
corr_matrix = np.corrcoef(sig_matrix)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
ax.set_xticks(range(len(top_genes)))
ax.set_yticks(range(len(top_genes)))
ax.set_xticklabels(top_genes, rotation=90, fontsize=8)
ax.set_yticklabels(top_genes, fontsize=8)
plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Perturbation signature correlations — top 20 genes")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "06_signature_correlations.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Clusters of correlated perturbations suggest shared biology (same pathway, complex, etc.)")

## 6. Save annotated AnnData

In [ ]:
# Add assignment confidence placeholder (in a real dataset this comes from pertpy)
adata.obs["assignment_confidence"] = np.where(
    adata.obs["is_ntc"], 1.0,  # NTC cells are always confidently assigned
    0.9  # Placeholder; in practice derived from guide UMI purity
)

out_path = os.path.join(DATA_DIR, "norman_assigned.h5ad")
adata.write_h5ad(out_path)
print(f"Saved to {out_path}")
print("obs columns added: gene_target, is_ntc, is_dual, n_targets, assignment_confidence")

## Key takeaways

1. Guide assignment converts a cells × guides UMI matrix into per-cell labels using either a UMI threshold or a probabilistic model
2. `pertpy.tl.GuideRNAAssignment` implements a Poisson mixture model that is more robust than a fixed threshold
3. Guide purity score (top guide UMI / total) is the primary multiplet indicator — exclude cells with purity < 0.75
4. Guide concordance (correlation of perturbation signatures from two guides targeting the same gene) is your primary on-target vs. off-target discriminator
5. Norman 2019 dual-guide cells (containing `+`) are retained for genetic interaction analysis in notebook 11

**Next:** [07_normalization_dimred.ipynb](07_normalization_dimred.ipynb)